In [1]:
import pandas as pd

In [2]:
records_per_ai = 300
records_human = records_per_ai * 5
objects = []

In [3]:
source_df = pd.read_csv('../dataset/sipora_title.csv')
endpoints: dict[int, None | str] = {}

for source_df in source_df.itertuples():
    if endpoints.get(source_df.year) is None:
        endpoints[source_df.year] = source_df.link
    else:
        continue

print(endpoints)

{2016: 'https://sipora.polije.ac.id/53207/', 2017: 'https://sipora.polije.ac.id/16352/', 2018: 'https://sipora.polije.ac.id/23207/', 2019: 'https://sipora.polije.ac.id/23546/', 2020: 'https://sipora.polije.ac.id/2648/'}


In [4]:
df = pd.read_csv("../dataset/sipora_content_human.csv")
df.head()

,link,content,type,lang,label
0,https://sipora.polije.ac.id/53207/,Bayam merupakan tumbuhan yang biasa ditanam un...,abstract,id,human
1,https://sipora.polije.ac.id/53207/,Kue kering merupakan salah satu jenis dari mak...,abstract,id,human
2,https://sipora.polije.ac.id/53207/,Pemasaran produk kue kering bayam dilakukan s...,abstract,id,human
3,https://sipora.polije.ac.id/53207/,Bayam merupakan tumbuhan yang biasa ditanam un...,content,id,human
4,https://sipora.polije.ac.id/53207/,Bayam juga sangat kaya akan antioksidan baik u...,content,id,human


In [5]:
filtered_data = None
count_per_year = int(records_human / len(endpoints))
column = ['link', 'content', 'type', 'lang', 'label']

for year in endpoints:
    endpoint = endpoints[year]
    index = df[df['link'] == endpoint].index
    filtered_items = df.iloc[index[0]:index[0]+count_per_year]
    
    if filtered_data is None:
        filtered_data = filtered_items
    else:
        filtered_data = pd.concat([filtered_data, filtered_items], ignore_index=True)

human_df = filtered_data
human_df.count()

link       1500
content    1500
type       1500
lang       1500
label      1500
dtype: int64

In [6]:
dfs = [
    pd.read_csv("../dataset/sipora_content_chatgpt.csv"),
    pd.read_csv("../dataset/sipora_content_claude.csv"),
    pd.read_csv("../dataset/sipora_content_gemini.csv"),
    pd.read_csv("../dataset/sipora_content_grok.csv"),
    pd.read_csv("../dataset/sipora_content_qwen.csv"),
]

ai_df = pd.concat([df for df in dfs], ignore_index=True)
ai_df.count()

link       1500
content    1500
type       1500
lang       1500
label      1500
dtype: int64

In [7]:
!rm -rf ../dataset/cleaned

In [10]:
combined_df = (pd
    .concat([human_df, ai_df], ignore_index=True)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
combined_df.count()

link       3000
content    3000
type       3000
lang       3000
label      3000
dtype: int64

In [12]:
from pathlib import Path

Path('../dataset/cleaned').mkdir(exist_ok=True)

combined_df.to_csv('../dataset/cleaned/contents.csv')